# 02 · 赛程爬虫 + 批量入库

## 这个 notebook 在做什么？

01 解决了「**单场**怎么爬」，02 解决「**全季**怎么爬」。

中间需要一个桥梁——**赛程接口**：从 `league_id`（赛事 ID）出发，拉到这个赛季的**所有 battle_id**，然后调 01 写好的 `fetch_battle()` 批量爬。

## 为什么要分成两个 notebook？

你之前 D 盘的项目，单场逻辑和批量逻辑混在一个文件里，结果：
- 改单场逻辑要改批量代码
- 改批量逻辑要担心影响单场
- 调试时要么全跑要么全不跑，没法局部验证

**关注点分离**——单场归单场（已沉淀到 `api_client.py`），赛程归赛程（这个 notebook 处理）。

## 跑完 notebook，你应该收获什么？

1. ✅ `fetch_schedule(league_id)` 函数：拉某赛季所有 battle_id
2. ✅ `data/processed/schedule.csv`：赛程总表（一行 = 一场比赛）
3. ✅ `data/raw/` 下囤了几十到上百个原始 JSON
4. ✅ 失败列表（哪些场没爬下来，为什么）

## 工作前提

> notebook 00 步骤 8 必须已经完成——**你已经知道赛程接口长什么样**（SSR 还是异步？URL 是什么？）。
> 如果还没，回去补这一步。

---
## 步骤 1 · 准备工作

### 思路

这次要跨文件夹 import `src` 下的模块。Python 的 import 机制有个坑：notebook 默认只能 import **同目录或 site-packages 里**的东西。

要想 import `src.crawler.api_client`，就得告诉 Python：「项目根目录也是个搜索路径」。

```python
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
```

`Path('..')` = 上一级（项目根目录），`.resolve()` = 转成绝对路径。这样 `from src.xxx import yyy` 就能用了。

In [28]:
# TODO 1.1：导包 + 加 sys.path

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))#sys.path是列表，0就是插在最前面。

# 然后导入：requests, json, pandas, tqdm.auto.tqdm
import requests
import json
import pandas as pd
from tqdm import tqdm



In [29]:
# TODO 1.2：从 src 导入 fetch_battle 并验证
# 调一次确保还能跑（顺便看缓存命中是否生效——不应该再走网络）

# from src.crawler.api_client import fetch_battle
# data = fetch_battle('1038107152_18_1742644777')
# print(data['code'])

from src.crawler.api_client import fetch_battle
data = fetch_battle('736117264_10_1777089898')


2026-05-09 23:26:55 | src.crawler.api_client | INFO | [命中缓存，736117264_10_1777089898]


---
## 步骤 2 · 写赛程爬虫

### 思路

根据你 notebook 00 步骤 8 的结论，分两条路：

**情况 A：赛程页是 SSR（HTML 里直接有数据）**

```python
import re
html = requests.get(schedule_url).text
# 用正则或 BeautifulSoup 抠出比赛信息
```

**情况 B：赛程是异步 JSON 接口**

```python
resp = requests.get(api_url, headers=headers)
data = resp.json()
# 一般 data['data']['list'] 之类的路径里能找到比赛列表
```

### 目标产出

不管哪种来源，最终都要产出**统一格式**的 list：

```python
[
    {
        'battle_id': '1038107152_18_1742644777',
        'match_id': '2026042501',          # 一场比赛的 ID（包含多局）
        'team_a_name': 'AG超玩会',
        'team_b_name': 'eStar',
        'start_time': '2026-04-25 18:00',
        'game_seq': 1,                      # 这是 BO 系列的第几局
        ...
    },
    ...
]
```

### 关键点

> **一场比赛 = 多局 = 多个 battle_id**。BO5 一场最多 5 个 battle_id，BO7 最多 7 个。要把每局都展开成一行，因为 `fetch_battle` 是按 battle_id 拉数据的。

In [30]:
# TODO 2.1：写 fetch_schedule(league_id) 函数
# 输入：赛事 ID（int），如 20260002
# 输出：list of dict，每个 dict 至少含 battle_id / match_id / 两支队 / 时间
#
# 实现思路（根据你 00 步骤 8 的结论选一种）：
#   - 路径 A（SSR）：抠 HTML
#   - 路径 B（异步）：直接调那个 JSON 接口
#
# 提示：
#   - 异步接口的 Headers 通常和单场接口不一样，要 F12 重新看
#   - 返回的 JSON 嵌套很深，多用 print 探索结构
#   - 用 pd.json_normalize 一行展平，比手动 dict 推导快
HEADERS = {
        "User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/128.0.0.0 Mobile Safari/537.36 Edg/128.0.0.0",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://pvp.qq.com/",
        "Origin": "https://pvp.qq.com",
    }
def fetch_league(league_id):
	#赛程url
    league_url = f"https://prod.comp.smoba.qq.com/leaguesite/matches/open?league_id={league_id}"

    league_list = []

    response = requests.get(url=league_url, headers=HEADERS,timeout=15)
    data_league = response.json()

    for i in range(len(data_league['results'])):
        league_list.append(data_league['results'][i])

    return league_list

In [31]:
fetch_league(20260002)

[{'match_id': '2026042501',
  'league_id': '20260002',
  'camp1': {'team_id': '10005',
   'team_name': 'KSG',
   'team_icon': 'https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89c5cf7e04d7f83534f.png',
   'is_win': False,
   'score': 1,
   'rank': 0,
   'team_abbreviation': 'KSG'},
  'camp2': {'team_id': '10017',
   'team_name': '广州TTG',
   'team_icon': 'https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823db5c8c44fae6abe.png',
   'is_win': True,
   'score': 3,
   'rank': 0,
   'team_abbreviation': 'TTG'},
  'bo': 5,
  'win_camp': 2,
  'status': 2,
  'start_time': '2026-04-25 12:00:00',
  'end_time': '2026-04-25 14:12:39',
  'match_address': '北京',
  'match_desc': '',
  'match_stage_seq': 0,
  'match_stage_name': 'dbtts',
  'match_stage_desc': '单败淘汰赛',
  'cc_match_id': 'KCC2026M1W1D1',
  'match_battle_video_list': [{'battle_id': '736117264_10_1777089898',
    'battle_seq': 1,
    'video_list': [{'video_channel': 'tencent_video',
      'video_id': 'e3237udgb2z',
      'video_url': 'https://v.qq.com/

In [32]:
# TODO 2.2：测试 + 转 DataFrame
# schedule_list = fetch_schedule(20260002)
# schedule_df = pd.DataFrame(schedule_list)
# print(schedule_df.shape)
# schedule_df.head()
#
# 自检：
#   - 行数对不对？挑战者杯 2026 大概 16 队 × 单循环 ≈ 几十场
#   - battle_id 是不是唯一？（df['battle_id'].duplicated().sum() 应该是 0）
#   - 时间字段是字符串还是时间戳？


schedule_list = fetch_league(20260002)
schedule_df = pd.DataFrame(schedule_list)

In [33]:
schedule_df.shape

(38, 16)

In [34]:
schedule_df.head()

,match_id,league_id,camp1,camp2,bo,win_camp,status,start_time,end_time,match_address,match_desc,match_stage_seq,match_stage_name,match_stage_desc,cc_match_id,match_battle_video_list
0,2026042501,20260002,"{'team_id': '10005', 'team_name': 'KSG', 'team...","{'team_id': '10017', 'team_name': '广州TTG', 'te...",5,2,2,2026-04-25 12:00:00,2026-04-25 14:12:39,北京,,0,dbtts,单败淘汰赛,KCC2026M1W1D1,"[{'battle_id': '736117264_10_1777089898', 'bat..."
1,2026042502,20260002,"{'team_id': '10002', 'team_name': '上海EDG.M', '...","{'team_id': '12401', 'team_name': '青训潜渊', 'tea...",5,1,2,2026-04-25 14:30:00,2026-04-25 16:51:32,北京,,0,dbtts,单败淘汰赛,KCC2026M1W1D2,"[{'battle_id': '736117264_14_1777099889', 'bat..."
2,2026042503,20260002,"{'team_id': '12402', 'team_name': '重庆科技大学', 't...","{'team_id': '10910', 'team_name': 'WST', 'team...",5,2,2,2026-04-25 18:30:00,2026-04-25 19:51:38,北京,,0,dbtts,单败淘汰赛,KCC2026M1W1D3,"[{'battle_id': '736117264_18_1777113272', 'bat..."
3,2026042504,20260002,"{'team_id': '12403', 'team_name': '阿里巴巴PKQ', '...","{'team_id': '10031', 'team_name': '杭州LGD.NBW',...",5,2,2,2026-04-25 21:00:00,2026-04-25 21:38:21,北京,,0,dbtts,单败淘汰赛,KCC2026M1W1D4,"[{'battle_id': '736117264_22_1777120389', 'bat..."
4,2026042601,20260002,"{'team_id': '10903', 'team_name': '桐乡情久', 'tea...","{'team_id': '10028', 'team_name': '长沙TES.A', '...",5,1,2,2026-04-26 12:00:00,2026-04-26 14:28:56,北京,,0,dbtts,单败淘汰赛,KCC2026M1W2D1,"[{'battle_id': '736117264_26_1777176389', 'bat..."


In [35]:
schedule_df.tail()

,match_id,league_id,camp1,camp2,bo,win_camp,status,start_time,end_time,match_address,match_desc,match_stage_seq,match_stage_name,match_stage_desc,cc_match_id,match_battle_video_list
33,2026051401,20260002,"{'team_id': '10000', 'team_name': '待定', 'team_...","{'team_id': '10000', 'team_name': '待定', 'team_...",7,0,0,2026-05-14 19:00:00,,北京,,0,sbtts,双败淘汰赛,KCC2026M3W7D1,[]
34,2026051501,20260002,"{'team_id': '10000', 'team_name': '待定', 'team_...","{'team_id': '10000', 'team_name': '待定', 'team_...",7,0,0,2026-05-15 19:00:00,,北京,,0,sbtts,双败淘汰赛,KCC2026M3W8D1,[]
35,2026051601,20260002,"{'team_id': '10000', 'team_name': '待定', 'team_...","{'team_id': '10000', 'team_name': '待定', 'team_...",7,0,0,2026-05-16 19:00:00,,北京,,0,sbtts,双败淘汰赛,KCC2026M3W9D1,[]
36,2026051701,20260002,"{'team_id': '10000', 'team_name': '待定', 'team_...","{'team_id': '10000', 'team_name': '待定', 'team_...",7,0,0,2026-05-17 19:00:00,,北京,,0,sbtts,双败淘汰赛,KCC2026M3W10D1,[]
37,2026052301,20260002,"{'team_id': '10000', 'team_name': '待定', 'team_...","{'team_id': '10000', 'team_name': '待定', 'team_...",7,0,0,2026-05-23 16:00:00,,上海,,0,js,决赛,KCC2026M4W1D1,[]


In [36]:
# TODO 2.3：保存赛程总表
# 路径：../data/processed/schedule.csv
# 编码：utf-8-sig （Excel 打开不乱码）
# index=False
# 思考：为什么把它放 processed/ 而不是 raw/？
# 提示：raw 是「接口原样」，processed 是「我整理过的结构化表」

schedule_root = Path("../data/processed").resolve()

path = f"{schedule_root}/schedule.csv"

schedule_df.to_csv(path, index=False,encoding='utf-8-sig')

💡 **观察**：

- 一共拉到多少条 battle_id？多少场 match？
- 时间跨度是从什么时候到什么时候？
- `match_id` 和 `battle_id` 的关系是什么？（一对多？数量比？）
- 有没有「未来场次」（还没打的）？模型只能用「已打完」的，需要过滤吗？

---
## 步骤 3 · 批量爬全部比赛

### 思路

拿到所有 battle_id 后，就是个简单的循环。但**不能简单的 `for + fetch_battle`**——要做几件事保命：

| 设计点 | 怎么做 | 为什么 |
|--------|--------|--------|
| 进度可见 | `tqdm` 包一层 | 几百条要跑半小时，没进度条会以为卡住 |
| 单条失败不影响整体 | `try/except` 包 fetch_battle 调用 | 一条失败就 raise = 前面跑过的白跑 |
| 失败要记录 | 用 list 收集 failed_ids | 跑完知道哪些要补爬 |
| 结果分类统计 | 成功 / 业务失败 / 技术失败 | 区别对待 |

### 反爬 vs 进度的取舍

`fetch_battle` 已经内置了 1-2 秒延时。100 场比赛 × 1.5 秒 ≈ 150 秒 ≈ 2.5 分钟。

第二次跑全部命中缓存，秒级完成。所以**第一次慢点没事**——之后修代码、调试都很快。

In [37]:
# TODO 3.1：批量爬
# 框架：
#   battle_ids = schedule_df['battle_id'].unique().tolist()
#   success, failed = [], []
#   for bid in tqdm(battle_ids):
#       try:
#           data = fetch_battle(bid)
#           if data is not None:
#               success.append(bid)
#           else:
#               failed.append((bid, 'fetch returned None'))
#       except Exception as e:
#           failed.append((bid, str(e)))



In [38]:
def fetch_match(match_id) :
	match_url = f"https://prod.comp.smoba.qq.com/leaguesite/match/battles/open?match_id={match_id}"
	battle_list = []
	response = requests.get(url=match_url, headers=HEADERS, timeout=20)
	data_match = response.json()
	for i in range(len(data_match['results'])):
		battle_list.append(data_match['results'][i]['battle_id'])
	return battle_list

In [39]:
schedule_df["match_id"].unique()

<StringArray>
['2026042501', '2026042502', '2026042503', '2026042504', '2026042601',
 '2026042602', '2026042603', '2026042604', '2026042701', '2026042702',
 '2026042703', '2026042704', '2026042801', '2026042802', '2026042803',
 '2026042804', '2026043001', '2026043002', '2026050101', '2026050102',
 '2026050201', '2026050202', '2026050301', '2026050302', '2026050801',
 '2026050802', '2026050901', '2026050902', '2026051001', '2026051002',
 '2026051101', '2026051201', '2026051301', '2026051401', '2026051501',
 '2026051601', '2026051701', '2026052301']
Length: 38, dtype: str

In [40]:
success = []
failed = []
battle_list = []

completed = schedule_df[schedule_df["status"] == 2]#已经完赛的
match_ids = completed["match_id"].unique().tolist()

In [42]:
for match_id in match_ids:
	battle_list.append(fetch_match(match_id))

In [43]:
battle_list

[['736117264_10_1777089898',
  '736117264_11_1777091838',
  '736117264_12_1777093747',
  '736117264_13_1777095778'],
 ['736117264_14_1777099889',
  '736117264_15_1777101864',
  '736117264_16_1777103765',
  '736117264_17_1777105595'],
 ['736117264_18_1777113272',
  '736117264_19_1777114949',
  '736117264_21_1777117059'],
 ['736117264_22_1777120389',
  '736117264_23_1777122032',
  '736117264_24_1777123595'],
 ['736117264_26_1777176389',
  '736117264_27_1777178528',
  '736117264_28_1777180478',
  '736117264_29_1777182144',
  '736117264_30_1777183470'],
 ['736117264_31_1777187214',
  '736117264_32_1777188987',
  '736117264_33_1777190822',
  '736117264_34_1777192390',
  '736117264_35_1777194471'],
 ['736117264_36_1777201256',
  '736117264_37_1777203203',
  '736117264_38_1777205445',
  '736117264_39_1777207738',
  '736117264_40_1777209524'],
 ['736117264_41_1777214738',
  '736117264_42_1777216972',
  '736117264_43_1777218912',
  '736117264_44_1777220644',
  '736117264_45_1777222317'],
 ['736

In [44]:
for list_id in battle_list:
	for battle_id in tqdm(list_id):

		try:
			data = fetch_battle(battle_id)
			if data is None:
				if battle_id not in failed:
					failed.append((battle_id, 'fetch returned None'))
			else:
				if battle_id not in success:
					success.append(battle_id)
		except Exception as e:
			if battle_id not in failed:
				failed.set((battle_id, str(e)))


  0%|          | 0/4 [00:00<?, ?it/s]

2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_10_1777089898]
2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_11_1777091838]
2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_12_1777093747]
2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_13_1777095778]


  0%|          | 0/4 [00:00<?, ?it/s]

2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_14_1777099889]
2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_15_1777101864]
2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_16_1777103765]
2026-05-09 23:29:29 | src.crawler.api_client | INFO | [命中缓存，736117264_17_1777105595]


  0%|          | 0/3 [00:00<?, ?it/s]

...

 60%|██████    | 3/5 [00:20<00:13,  6.74s/it]

2026-05-09 23:30:50 | src.crawler.api_client | INFO | 已查询到数据，正在写入数据
2026-05-09 23:30:51 | src.crawler.api_client | INFO | 数据保存成功，路径：D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw\736117264_37_1778330358.json


 80%|████████  | 4/5 [00:24<00:05,  5.59s/it]

2026-05-09 23:30:53 | src.crawler.api_client | INFO | 已查询到数据，正在写入数据
2026-05-09 23:30:55 | src.crawler.api_client | INFO | 数据保存成功，路径：D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw\736117264_38_1778331975.json


100%|██████████| 5/5 [00:27<00:00,  5.57s/it]


append会出现重复值

In [45]:
# TODO 3.2：跑完后统计 + 保存失败列表
# 1. 成功率：len(success) / 总数
# 2. 文件数核对：len(list(Path('../data/raw').glob('*.json'))) 应该 ≈ len(success)
# 3. 失败列表存成 CSV：../output/failed_ids.csv（方便日后补爬）
success

['736117264_10_1777089898',
 '736117264_11_1777091838',
 '736117264_12_1777093747',
 '736117264_13_1777095778',
 '736117264_14_1777099889',
 '736117264_15_1777101864',
 '736117264_16_1777103765',
 '736117264_17_1777105595',
 '736117264_18_1777113272',
 '736117264_19_1777114949',
 '736117264_21_1777117059',
 '736117264_22_1777120389',
 '736117264_23_1777122032',
 '736117264_24_1777123595',
 '736117264_26_1777176389',
 '736117264_27_1777178528',
 '736117264_28_1777180478',
 '736117264_29_1777182144',
 '736117264_30_1777183470',
 '736117264_31_1777187214',
 '736117264_32_1777188987',
 '736117264_33_1777190822',
 '736117264_34_1777192390',
 '736117264_35_1777194471',
 '736117264_36_1777201256',
 '736117264_37_1777203203',
 '736117264_38_1777205445',
 '736117264_39_1777207738',
 '736117264_40_1777209524',
 '736117264_41_1777214738',
 '736117264_42_1777216972',
 '736117264_43_1777218912',
 '736117264_44_1777220644',
 '736117264_45_1777222317',
 '736117264_48_1777262688',
 '736117264_49_17772

In [46]:
failed

[]

In [47]:
battle_id_list = []
for list_id in battle_list:
	for battle_id in list_id:
		battle_id_list.append(battle_id)


In [48]:
battle_id_list

['736117264_10_1777089898',
 '736117264_11_1777091838',
 '736117264_12_1777093747',
 '736117264_13_1777095778',
 '736117264_14_1777099889',
 '736117264_15_1777101864',
 '736117264_16_1777103765',
 '736117264_17_1777105595',
 '736117264_18_1777113272',
 '736117264_19_1777114949',
 '736117264_21_1777117059',
 '736117264_22_1777120389',
 '736117264_23_1777122032',
 '736117264_24_1777123595',
 '736117264_26_1777176389',
 '736117264_27_1777178528',
 '736117264_28_1777180478',
 '736117264_29_1777182144',
 '736117264_30_1777183470',
 '736117264_31_1777187214',
 '736117264_32_1777188987',
 '736117264_33_1777190822',
 '736117264_34_1777192390',
 '736117264_35_1777194471',
 '736117264_36_1777201256',
 '736117264_37_1777203203',
 '736117264_38_1777205445',
 '736117264_39_1777207738',
 '736117264_40_1777209524',
 '736117264_41_1777214738',
 '736117264_42_1777216972',
 '736117264_43_1777218912',
 '736117264_44_1777220644',
 '736117264_45_1777222317',
 '736117264_48_1777262688',
 '736117264_49_17772

In [49]:
len(success)/len(battle_id_list)

0.774390243902439

In [50]:
len(list(Path("../data/raw").glob("*.json")))

127

In [51]:
pd.DataFrame(failed).to_csv("../output/failed.csv")

💡 **观察**：

- 成功率多少？低于 95% 就要看失败原因 100%
- 失败的 battle_id 集中在某些场次吗？还是随机分布？没有
- 大部分失败的原因是什么？（业务错误 vs 技术错误的比例）无
- 第一次跑总耗时多少？（用来评估爬虫的可扩展性）

---
## 步骤 4 · 沉淀到 src 模块

### 任务

把 `fetch_schedule()` 函数迁移到 `src/crawler/schedule_crawler.py`。

迁移规范（同 notebook 01 步骤 7）：
1. docstring 写清楚参数、返回值
2. 类型注解 `def fetch_schedule(league_id: int) -> list[dict]:`
3. 用 `from src.utils.logger import get_logger` 拿 logger
4. URL 和 Headers 抽到 `src/config.py`（如果赛程接口的 Headers 跟单场不一样，单独加常量）

迁移完在下面验证：

In [52]:
# TODO 4.1：从 src 导入 fetch_schedule 验证
# from src.crawler.schedule_crawler import fetch_schedule
# rows = fetch_schedule(20260002)
# print(len(rows), rows[0])

from src.crawler.schedule_crawler import fetch_schedule

rows =fetch_schedule(20260002)
print(len(rows),rows.iloc[0])

2026-05-09 23:31:05 | src.crawler.schedule_crawler | INFO | 拉取赛程成功：38 场 match
2026-05-09 23:31:05 | src.crawler.schedule_crawler | INFO | 赛程已保存至 D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\processed\schedule.csv
38 match_id                                                          2026042501
league_id                                                           20260002
camp1                      {'team_id': '10005', 'team_name': 'KSG', 'team...
camp2                      {'team_id': '10017', 'team_name': '广州TTG', 'te...
bo                                                                         5
win_camp                                                                   2
status                                                                     2
start_time                                               2026-04-25 12:00:00
end_time                                                 2026-04-25 14:12:39
match_address                                                             北京
match_desc             

---
## ✅ 完成自检

- [ ] `fetch_schedule()` 能拉到完整赛程
- [ ] `data/processed/schedule.csv` 已生成，可以用 Excel 打开看
- [ ] 批量爬完成，成功率 ≥ 95%
- [ ] 失败的 battle_id 写到了 `output/failed_ids.csv`
- [ ] `data/raw/` 下囤了几十个原始 JSON
- [ ] `src/crawler/schedule_crawler.py` 迁移完成

## 🎯 完成后跟我汇报

```
1. 我做了什么：______
2. 我的思考：______
   - 这个赛事一共多少场？
   - 失败的占比 / 失败的原因
3. 我想确认：______
```